In [1]:
import os, json, contextlib, shutil
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm import tqdm
import pandas as pd
from scipy.ndimage import label as cc_label
from scipy.stats import spearmanr
from medpy.metric.binary import hd95 as medpy_hd95
import cv2
import torch
from ultralytics import YOLO

SPLITS_DIR = Path(r"D:\master_experiments\data\splits\BraTS2020_Splits")
META_PATH  = SPLITS_DIR / "splits_metadata.json"

YOLOSEG_BASE    = Path(r"D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020")
YOLOSEG_CKPT    = YOLOSEG_BASE / "checkpoints"
YOLOSEG_PRED    = YOLOSEG_BASE / "predictions_test"
YOLOSEG_LOGS    = YOLOSEG_BASE / "logs"
YOLOSEG_DATA    = YOLOSEG_BASE / "yolo_dataset"
YOLOSEG_WEIGHTS = YOLOSEG_BASE / "weights" 

for p in [YOLOSEG_BASE, YOLOSEG_CKPT, YOLOSEG_PRED, YOLOSEG_LOGS, YOLOSEG_WEIGHTS,
          YOLOSEG_DATA / "images" / "train", YOLOSEG_DATA / "images" / "val",
          YOLOSEG_DATA / "labels" / "train", YOLOSEG_DATA / "labels" / "val"]:
    p.mkdir(parents=True, exist_ok=True)


MODS       = ["flair", "t1", "t1ce", "t2"]
YOLO_MODS  = ["t1ce", "flair", "t2"]   # 3 canais (R, G, B) usados pelo YOLO
N_CLASSES  = 4                       
YOLO_NC    = 3                      
YOLO_NAMES = {0: "NCR", 1: "ED", 2: "ET"}
YOLO_IMGSZ = 256
YOLO_CONF  = 0.25

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
NW      = 0 if os.name == "nt" else 4

print("Device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 SUPER | VRAM: 12.9 GB


In [2]:
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

train_ids = meta["ids"]["train"]
val_ids = meta["ids"]["val"]
test_ids = meta["ids"]["test"]

train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("Counts:", len(train_ids), len(val_ids), len(test_ids))
print("OK: sem repetição entre splits")

Counts: 245 52 53
OK: sem repetição entre splits


In [3]:
# Funções auxiliares

def case_dir(split_name: str, case_id: str) -> Path:
    return SPLITS_DIR / split_name / case_id

def find_file(folder: Path, key: str) -> Path:
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists():
            return cand
    cands = sorted(list(folder.glob(f"*{key}*.nii*")))
    if key == "t1":
        cands = [c for c in cands if "t1ce" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def load_arr(p) -> np.ndarray:
    return np.squeeze(np.asanyarray(nib.load(str(p)).dataobj))


def load_brats_seg(path) -> np.ndarray:
    data = load_arr(path).astype(np.int16)
    data[data == 4] = 3
    return data

def pick_best_slice(seg, axis=2) -> int:
    counts = np.sum(seg > 0, axis=tuple(i for i in range(3) if i != axis))
    return int(np.argmax(counts))

for cid in train_ids[:3]:
    d = case_dir("train", cid)
    assert d.exists()
    _ = [find_file(d, m) for m in MODS]
    _ = find_file(d, "seg")
print("Helpers OK")

Helpers OK


In [4]:
# Normalização e conversão de máscaras para polígonos YOLO

def norm01(x, p1=1, p99=99) -> np.ndarray:
    x = x.astype(np.float32)

    finite_mask = np.isfinite(x)
    nonzero_mask = x != 0
    valid_mask = finite_mask & nonzero_mask
    if valid_mask.sum() < 16:
        valid_mask = finite_mask

    vals = x[valid_mask]
    if vals.size == 0:
        return np.zeros_like(x, dtype=np.float32)

    lo, hi = np.percentile(vals, [p1, p99])

    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)

    x = np.clip((x - lo) / (hi - lo), 0, 1)
    x[~finite_mask] = 0
    return x.astype(np.float32)


def slice_to_rgb_uint8(d: Path, z: int) -> np.ndarray:
    chans = [norm01(load_arr(find_file(d, m)))[:, :, z] for m in YOLO_MODS]
    rgb = np.stack(chans, axis=-1)
    rgb = np.clip(rgb, 0, 1)
    return np.round(rgb * 255).astype(np.uint8)


def mask_to_yolo_polygons(seg2d: np.ndarray):
    # Converte máscara densa 2D BraTS {0,1,2,3} para polígonos.

    seg2d = np.ascontiguousarray(seg2d)
    H, W = seg2d.shape
    polys = []

    for brats_cls in [1, 2, 3]:
        m = np.ascontiguousarray((seg2d == brats_cls).astype(np.uint8))

        if m.sum() == 0:
            continue

        contours, _ = cv2.findContours(
            m,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for cnt in contours:
            if cnt is None or len(cnt) < 3:
                continue

            if cv2.contourArea(cnt) < 1.0:
                continue

            pts = cnt.reshape(-1, 2).astype(np.float32)

            if len(np.unique(pts, axis=0)) < 3:
                continue

            pts[:, 0] = np.clip(pts[:, 0] / float(W), 0.0, 1.0)
            pts[:, 1] = np.clip(pts[:, 1] / float(H), 0.0, 1.0)

            polys.append((brats_cls - 1, pts.flatten().tolist()))
            
    return polys
    
print("funções auxiliares ok")

funções auxiliares ok


In [5]:
# Exporta o dataset 2D no formato YOLO

NEG_RATE = 0.33
FORCE_REEXPORT_YOLO = True

if FORCE_REEXPORT_YOLO:
    for p in [
        YOLOSEG_DATA / "images" / "train",
        YOLOSEG_DATA / "images" / "val",
        YOLOSEG_DATA / "labels" / "train",
        YOLOSEG_DATA / "labels" / "val",
    ]:
        shutil.rmtree(p, ignore_errors=True)
        p.mkdir(parents=True, exist_ok=True)


def export_case_to_yolo(cid: str, src_split: str, dst_split: str, rng: np.random.Generator):
    d = case_dir(src_split, cid)

    seg_3d = load_brats_seg(find_file(d, "seg"))
    chans_3d = [norm01(load_arr(find_file(d, m))) for m in YOLO_MODS]

    H, W, D = seg_3d.shape
    n_saved = 0

    for z in range(D):
        seg2d = seg_3d[:, :, z]
        has_lesion = (seg2d > 0).any()

        if not has_lesion and rng.random() > NEG_RATE:
            continue

        img_path = YOLOSEG_DATA / "images" / dst_split / f"{cid}_z{z:03d}.png"
        lbl_path = YOLOSEG_DATA / "labels" / dst_split / f"{cid}_z{z:03d}.txt"

        rgb = np.stack([c[:, :, z] for c in chans_3d], axis=-1)
        rgb = np.clip(rgb, 0, 1)
        rgb_u8 = np.round(rgb * 255).astype(np.uint8)

        cv2.imwrite(str(img_path), cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2BGR))

        polys = mask_to_yolo_polygons(seg2d) if has_lesion else []

        with open(lbl_path, "w", encoding="utf-8") as f:
            for cls_id, pts in polys:
                f.write(f"{cls_id} " + " ".join(f"{v:.6f}" for v in pts) + "\n")

        n_saved += 1

    return n_saved

rng = np.random.default_rng(42)

tot_tr = 0
for cid in tqdm(train_ids, desc="Export train -> YOLO"):
    tot_tr += export_case_to_yolo(cid, "train", "train", rng)

tot_va = 0
for cid in tqdm(val_ids, desc="Export val -> YOLO"):
    tot_va += export_case_to_yolo(cid, "val", "val", rng)

data_yaml_path = YOLOSEG_DATA / "data.yaml"
yaml_str = (
    f"path: {YOLOSEG_DATA.as_posix()}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"nc: {YOLO_NC}\n"
    f"names:\n"
    + "".join(f"  {k}: {v}\n" for k, v in YOLO_NAMES.items())
)

data_yaml_path.write_text(yaml_str, encoding="utf-8")

print(f"Slices train: {tot_tr} | val: {tot_va}")
print("data.yaml ->", data_yaml_path)

Export val -> YOLO: 100%|██████████| 52/52 [00:33<00:00,  1.56it/s]

Slices train: 23308 | val: 5053
data.yaml -> D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\yolo_dataset\data.yaml


In [6]:
# labels YOLO exportados

def audit_yolo_labels(split_name: str):
    labels_dir = YOLOSEG_DATA / "labels" / split_name

    total_files = 0
    empty_files = 0
    total_instances = 0
    bad_lines = []

    class_counts = {k: 0 for k in YOLO_NAMES.keys()}

    for lbl_path in sorted(labels_dir.glob("*.txt")):
        total_files += 1
        lines = lbl_path.read_text(encoding="utf-8").strip().splitlines()

        if len(lines) == 0:
            empty_files += 1
            continue

        for line_idx, line in enumerate(lines, start=1):
            parts = line.strip().split()

            if len(parts) < 7:
                bad_lines.append((str(lbl_path), line_idx, "linha com poucos valores"))
                continue

            try:
                cls_id = int(float(parts[0]))
                coords = np.array([float(v) for v in parts[1:]], dtype=np.float32)
            except Exception:
                bad_lines.append((str(lbl_path), line_idx, "erro ao converter valores"))
                continue

            if cls_id not in YOLO_NAMES:
                bad_lines.append((str(lbl_path), line_idx, f"classe inválida: {cls_id}"))
                continue

            if len(coords) % 2 != 0:
                bad_lines.append((str(lbl_path), line_idx, "número ímpar de coordenadas"))
                continue

            if len(coords) < 6:
                bad_lines.append((str(lbl_path), line_idx, "polígono com menos de 3 pontos"))
                continue

            if np.any(coords < 0) or np.any(coords > 1):
                bad_lines.append((str(lbl_path), line_idx, "coordenadas fora de [0,1]"))
                continue

            class_counts[cls_id] += 1
            total_instances += 1

    print(f"\nAuditoria YOLO labels — split={split_name}")
    print(f"Arquivos .txt:             {total_files}")
    print(f"Arquivos vazios/background:{empty_files}")
    print(f"Instâncias anotadas:       {total_instances}")
    print(f"Contagem por classe:       {class_counts}")
    print(f"Linhas problemáticas:      {len(bad_lines)}")

    if bad_lines:
        print("\nExemplos de problemas:")
        for item in bad_lines[:10]:
            print(item)

    return {
        "split": split_name,
        "total_files": total_files,
        "empty_files": empty_files,
        "total_instances": total_instances,
        "class_counts": class_counts,
        "bad_lines": bad_lines,
    }

audit_train = audit_yolo_labels("train")
audit_val = audit_yolo_labels("val")


Auditoria YOLO labels — split=train
Arquivos .txt:             23308
Arquivos vazios/background:7518
Instâncias anotadas:       77869
Contagem por classe:       {0: 24405, 1: 37564, 2: 15900}
Linhas problemáticas:      0

Auditoria YOLO labels — split=val
Arquivos .txt:             5053
Arquivos vazios/background:1599
Instâncias anotadas:       17755
Contagem por classe:       {0: 5544, 1: 8570, 2: 3641}
Linhas problemáticas:      0


In [7]:
# ═══════════════════════════════════════════════════════════
#  Instanciação do YOLO-Seg (Ultralytics) para segmentação 2D multiclasse
# ═══════════════════════════════════════════════════════════
#
#  weights = yolo11n-seg.pt → pretrained COCO seg (s = small)
#  task    = segment        → cabeca de mascaras de instancia
#  nc      = 3              → NCR, ED, ET (background = no detection)
#  imgsz   = 256            → slices BraTS 240x240 -> letterbox 256x256
#  ch_in   = 3              → R=T1ce, G=FLAIR, B=T2 (T1 omitido p/ usar pre-trained)
#
#  YOLO-Seg instance segmentation: cada componente conexo de
#  uma classe vira uma instancia. Ao reconstruir, unimos todas as instancias
#  da mesma classe (semantica) e pintamos por prioridade ED < NCR < ET.
#
# ═══════════════════════════════════════════════════════════

YOLO_WEIGHTS = "yolo11s-seg.pt"

_old_cwd = os.getcwd()
os.chdir(YOLOSEG_WEIGHTS)
try:
    model = YOLO(YOLO_WEIGHTS)
finally:
    os.chdir(_old_cwd)

n_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad) / 1e6

print("=" * 55)
print(f"  YOLO-Seg ({YOLO_WEIGHTS})  —  Configuração do modelo")
print("=" * 55)
print(f"  Canais de entrada (in_channels) : 3 (R=T1ce, G=FLAIR, B=T2)")
print(f"  Classes YOLO (foreground)       : {YOLO_NC} ({list(YOLO_NAMES.values())})")
print(f"  Imgsz                           : {YOLO_IMGSZ}")
print(f"  Device                          : {DEVICE}")
print(f"  Parâmetros treináveis           : {n_params:.2f} M")
print(f"  Pesos pretrained salvos em      : {YOLOSEG_WEIGHTS}")
print("=" * 55)

  YOLO-Seg (yolo11s-seg.pt)  —  Configuração do modelo
  Canais de entrada (in_channels) : 3 (R=T1ce, G=FLAIR, B=T2)
  Classes YOLO (foreground)       : 3 (['NCR', 'ED', 'ET'])
  Imgsz                           : 256
  Device                          : cuda
  Parâmetros treináveis           : 0.00 M
  Pesos pretrained salvos em      : D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\weights


In [8]:
# Loss do YOLO-Seg (composta, gerenciada internamente pelo Ultralytics):
# loss_total = box_loss (CIoU) + cls_loss (BCE) + dfl_loss (Dist Focal) + seg_loss (BCE de mask)

print("YOLO-Seg loss = CIoU + BCE_cls + DFL + BCE_mask  (gerenciada pelo Ultralytics).")

YOLO-Seg loss = CIoU + BCE_cls + DFL + BCE_mask  (gerenciada pelo Ultralytics).


In [9]:
# Treino YOLO-Seg com augmentations para RM multimodal

TRAIN_EPOCHS = 100
BATCH_SIZE   = 16
LR           = 1e-3

_old_cwd = os.getcwd()
os.chdir(YOLOSEG_WEIGHTS)

try:
    results = model.train(
        data         = str(data_yaml_path),
        epochs       = TRAIN_EPOCHS,
        imgsz        = YOLO_IMGSZ,
        batch        = BATCH_SIZE,
        optimizer    = "AdamW",
        lr0          = LR,
        lrf          = 0.01,
        cos_lr       = True,
        weight_decay = 1e-5,
        amp          = USE_AMP,
        workers      = NW,
        seed         = 42,
        device       = 0 if torch.cuda.is_available() else "cpu",
        project      = str(YOLOSEG_BASE),
        name         = "train",
        exist_ok     = True,
        save         = True,
        save_period  = -1,
        verbose      = True,
        hsv_h        = 0.0,
        hsv_s        = 0.0,
        hsv_v        = 0.0,
        mosaic       = 0.0,
        mixup        = 0.0,
        copy_paste   = 0.0,
        erasing      = 0.0,
        degrees      = 0.0,
        translate    = 0.05,
        scale        = 0.25,
        shear        = 0.0,
        perspective  = 0.0,
        flipud       = 0.0,
        fliplr       = 0.5,
    )

finally:
    os.chdir(_old_cwd)

src_best = YOLOSEG_BASE / "train" / "weights" / "best.pt"

if src_best.exists():
    shutil.copy2(src_best, YOLOSEG_CKPT / "yoloseg_best.pt")
    print(f"Best checkpoint -> {YOLOSEG_CKPT / 'yoloseg_best.pt'}")

results_csv = YOLOSEG_BASE / "train" / "results.csv"

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = [c.strip() for c in df_res.columns]

    train_loss_cols = [c for c in df_res.columns if c.startswith("train/") and "loss" in c]
    val_loss_cols   = [c for c in df_res.columns if c.startswith("val/")   and "loss" in c]

    tr_loss = df_res[train_loss_cols].sum(axis=1).tolist() if train_loss_cols else []
    va_loss = df_res[val_loss_cols].sum(axis=1).tolist()   if val_loss_cols   else []

    train_history = {"train": tr_loss, "val": va_loss}

    with open(YOLOSEG_LOGS / "train_history.json", "w", encoding="utf-8") as f:
        json.dump(train_history, f, indent=2)

    print("Treino concluído.")
else:
    train_history = {"train": [], "val": []}
    print("Aviso: results.csv nao encontrado.")

New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11

In [10]:
# Inferência YOLO com consistência de canais RGB/BGR

def slicewise_inference_yolo(model, vol_3rgb_3d: np.ndarray, n_classes: int,
                              imgsz: int = YOLO_IMGSZ, conf: float = YOLO_CONF) -> np.ndarray:

    _, H, W, D = vol_3rgb_3d.shape
    pred_3d = np.zeros((H, W, D), dtype=np.int16)

    priority = {0: 1, 1: 0, 2: 2}

    for z in range(D):
        rgb = np.stack([vol_3rgb_3d[c, :, :, z] for c in range(3)], axis=-1)
        rgb = np.clip(rgb, 0, 1)

        rgb_u8 = np.round(rgb * 255).astype(np.uint8)
        bgr_u8 = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2BGR)

        results = model.predict(
            source=bgr_u8,
            imgsz=imgsz,
            conf=conf,
            verbose=False,
            retina_masks=True,
            device=0 if torch.cuda.is_available() else "cpu"
        )

        r = results[0]

        if r.masks is None or len(r.masks.data) == 0:
            continue

        masks = r.masks.data.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)

        order = sorted(range(len(classes)), key=lambda i: priority.get(int(classes[i]), -1))
        label_2d = np.zeros((H, W), dtype=np.int16)

        for i in order:
            cls_yolo = int(classes[i])

            if cls_yolo not in YOLO_NAMES:
                continue

            cls_brats = cls_yolo + 1

            m = masks[i] > 0.5

            if m.shape != (H, W):
                m = cv2.resize(
                    m.astype(np.uint8),
                    (W, H),
                    interpolation=cv2.INTER_NEAREST
                ).astype(bool)
            label_2d[m] = cls_brats
        pred_3d[:, :, z] = label_2d
    return pred_3d

print("slicewise_inference_yolo")

slicewise_inference_yolo


In [11]:
# Carrega melhor checkpoint e gera predições no conjunto de teste

ckpt_path = YOLOSEG_CKPT / "yoloseg_best.pt"
model = YOLO(str(ckpt_path))

print("Melhor modelo YOLO-Seg carregado.")
print(f"\nGerando predições → {YOLOSEG_PRED}")

for cid in tqdm(test_ids, desc="Predição test"):
    out_path = YOLOSEG_PRED / f"{cid}.nii.gz"

    d = case_dir("test", cid)
    ref = nib.load(str(find_file(d, "flair")))

    chans_3d = [norm01(load_arr(find_file(d, m))) for m in YOLO_MODS]
    vol = np.stack(chans_3d, axis=0).astype(np.float32)

    pred = slicewise_inference_yolo(
        model,
        vol,
        n_classes=N_CLASSES,
        imgsz=YOLO_IMGSZ,
        conf=YOLO_CONF
    )
    nib.save(nib.Nifti1Image(pred.astype(np.int16), ref.affine, ref.header), str(out_path))

print("Predições concluídas.")

Melhor modelo YOLO-Seg carregado.

Gerando predições → D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\predictions_test


Predição test: 100%|██████████| 53/53 [02:51<00:00,  3.23s/it]

Predições concluídas.


In [12]:
# Dice global por caso

def get_pred_path(cid: str) -> Path:
    for ext in [".nii.gz", ".nii"]:
        p = YOLOSEG_PRED / f"{cid}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Pred não encontrada para {cid}")

def dice(a, b) -> float:
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

rows = []
for cid in tqdm(test_ids, desc="Dice por caso"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    d1 = dice(gt == 1, pr == 1)
    d2 = dice(gt == 2, pr == 2)
    d3 = dice(gt == 3, pr == 3)
    wt = dice(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))

    rows.append({"id": cid,
                 "dice_c1": d1, "dice_c2": d2, "dice_ET": d3,
                 "dice_WT": wt, "dice_TC": tc})

df = pd.DataFrame(rows)
df.to_csv(YOLOSEG_LOGS / "dice_test.csv", index=False)
df.describe()

Dice por caso: 100%|██████████| 53/53 [00:08<00:00,  6.48it/s]


,dice_c1,dice_c2,dice_ET,dice_WT,dice_TC
count,53.000000,53.000000,53.000000,53.000000,53.000000
mean,0.500964,0.770797,0.710769,0.845651,0.792138
std,0.334900,0.124417,0.204159,0.092759,0.156777
min,0.000000,0.419599,0.000000,0.459500,0.162563
25%,0.149258,0.671042,0.659748,0.821076,0.764789
50%,0.597811,0.816220,0.786966,0.866693,0.843768
75%,0.776576,0.864915,0.839283,0.902322,0.896147
max,0.950418,0.930680,0.921995,0.943338,0.937486


In [13]:
# HD95 (Hausdorff Distance 95th percentile) por caso — mesmo padrão do Dice global, por regiões

def hd95_score(gt_mask, pr_mask, voxelspacing=None):
    gt_b = gt_mask.astype(bool)
    pr_b = pr_mask.astype(bool)
    if not gt_b.any() and not pr_b.any():
        return 0.0
    if not gt_b.any() or not pr_b.any():
        return np.nan
    return float(medpy_hd95(pr_b, gt_b, voxelspacing=voxelspacing))


hd95_rows = []
for cid in tqdm(test_ids, desc="HD95 por caso"):
    gt_path = find_file(case_dir("test", cid), "seg")
    pr_path = get_pred_path(cid)

    gt = load_brats_seg(gt_path)
    pr = load_arr(pr_path).astype(np.int16)

    spacing = nib.load(str(gt_path)).header.get_zooms()[:3]

    h1  = hd95_score(gt == 1, pr == 1, voxelspacing=spacing)
    h2  = hd95_score(gt == 2, pr == 2, voxelspacing=spacing)
    h3  = hd95_score(gt == 3, pr == 3, voxelspacing=spacing)
    hwt = hd95_score(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)), voxelspacing=spacing)
    htc = hd95_score(((gt == 1) | (gt == 3)),   ((pr == 1) | (pr == 3)),   voxelspacing=spacing)

    hd95_rows.append({
        "id":       cid,
        "hd95_c1":  h1,
        "hd95_c2":  h2,
        "hd95_ET":  h3,
        "hd95_WT":  hwt,
        "hd95_TC":  htc,
    })

df_hd95 = pd.DataFrame(hd95_rows)
df_hd95.to_csv(YOLOSEG_LOGS / "hd95_test.csv", index=False)
df_hd95.describe()

HD95 por caso: 100%|██████████| 53/53 [04:41<00:00,  5.31s/it]


,hd95_c1,hd95_c2,hd95_ET,hd95_WT,hd95_TC
count,45.000000,53.000000,52.000000,53.000000,53.000000
mean,14.943477,7.983353,7.732389,9.137836,9.049634
std,20.359835,12.658914,18.126932,12.434673,16.652300
min,1.414214,1.732051,1.414214,3.464102,2.236068
25%,5.000000,3.000000,2.000000,5.385165,4.242641
50%,8.774964,4.582576,2.236068,6.782330,5.916080
75%,16.363063,6.708204,3.280776,8.062258,8.062258
max,127.659704,84.645140,114.630057,93.068522,124.201041


In [14]:
# Análise condicional 1/3: frequência de presença de cada classe no GT

presence_rows = []
for cid in tqdm(test_ids, desc="Presença GT"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    presence_rows.append({
        "id":     cid,
        "has_c1": bool((gt == 1).any()),
        "has_c2": bool((gt == 2).any()),
        "has_ET": bool((gt == 3).any()),
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
    })
pres_df = pd.DataFrame(presence_rows)
df_full = df.merge(pres_df, on="id")

df_presence = pd.DataFrame([
    {"classe":     col,
     "n_presente": int(df_full[col].sum()),
     "n_total":    len(df_full),
     "pct":        100.0 * df_full[col].sum() / len(df_full)}
    for col in ["has_c1", "has_c2", "has_ET", "has_WT", "has_TC"]
])
df_presence.round(2)

Presença GT: 100%|██████████| 53/53 [00:02<00:00, 25.13it/s]


,classe,n_presente,n_total,pct
0,has_c1,53,53,100.0
1,has_c2,53,53,100.0
2,has_ET,53,53,100.0
3,has_WT,53,53,100.0
4,has_TC,53,53,100.0


In [15]:
# Análise condicional 2/3: Dice somente nos casos em que a classe está presente no GT
pairs = [("dice_c1","has_c1"), ("dice_c2","has_c2"), ("dice_ET","has_ET"),
         ("dice_WT","has_WT"), ("dice_TC","has_TC")]
df_conditional = pd.DataFrame({
    col: df_full.loc[df_full[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional.to_csv(YOLOSEG_LOGS / "dice_conditional.csv", index=False)
print("\nDice condicional:")
df_conditional.describe().round(4)


Dice condicional:


,dice_c1,dice_c2,dice_ET,dice_WT,dice_TC
count,53.0000,53.0000,53.0000,53.0000,53.0000
mean,0.5010,0.7708,0.7108,0.8457,0.7921
std,0.3349,0.1244,0.2042,0.0928,0.1568
min,0.0000,0.4196,0.0000,0.4595,0.1626
25%,0.1493,0.6710,0.6597,0.8211,0.7648
50%,0.5978,0.8162,0.7870,0.8667,0.8438
75%,0.7766,0.8649,0.8393,0.9023,0.8961
max,0.9504,0.9307,0.9220,0.9433,0.9375


In [16]:
# Análise condicional 3/3: bimodalidade — % de Dice=0 e % de Dice≈1

df_bimodality = pd.DataFrame([
    {"metric":         col,
     "pct_dice_zero": float((df_full[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality.to_csv(YOLOSEG_LOGS / "dice_bimodality.csv", index=False)
df_bimodality.round(2)

,metric,pct_dice_zero,pct_dice_one
0,dice_c1,18.87,0.0
1,dice_c2,0.00,0.0
2,dice_ET,3.77,0.0
3,dice_WT,0.00,0.0
4,dice_TC,0.00,0.0


In [17]:
# Lesion-wise Dice (LWD) — métrica primária do BraTS 2023+

LW_MIN_SIZE = 50

def lesion_wise_dice(gt_mask: np.ndarray, pr_mask: np.ndarray,
                     min_size: int = LW_MIN_SIZE) -> float:
    structure = np.ones((3, 3, 3), dtype=int)
    if not gt_mask.any() and not pr_mask.any():
        return 1.0
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)
    dice_list, matched_pred = [], set()
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap_ids = np.unique(pr_lab[gt_l])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) == 0:
            dice_list.append(0.0)  # false negative
        else:
            pr_match = np.isin(pr_lab, overlap_ids)
            inter = np.logical_and(gt_l, pr_match).sum()
            denom = gt_l.sum() + pr_match.sum()
            dice_list.append(2.0 * inter / denom if denom > 0 else 0.0)
            matched_pred.update(int(p) for p in overlap_ids)
    for p in range(1, n_pr + 1):
        if p in matched_pred:
            continue
        if (pr_lab == p).sum() < min_size:
            continue
        dice_list.append(0.0)  # false positive
    return float(np.mean(dice_list)) if dice_list else 1.0


lw_rows = []
for cid in tqdm(test_ids, desc="Lesion-wise Dice"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    gt_wt = (gt==1)|(gt==2)|(gt==3);  pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3);          pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3);                  pr_et = (pr==3)

    lw_rows.append({"id":     cid,
                    "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
                    "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
                    "lwd_ET": lesion_wise_dice(gt_et, pr_et)})

lw_df = pd.DataFrame(lw_rows)
lw_df.to_csv(YOLOSEG_LOGS / "dice_lesionwise.csv", index=False)
lw_df[["lwd_WT", "lwd_TC", "lwd_ET"]].describe().round(4)

Lesion-wise Dice: 100%|██████████| 53/53 [00:44<00:00,  1.19it/s]


,lwd_WT,lwd_TC,lwd_ET
count,53.0000,53.0000,53.0000
mean,0.5601,0.7162,0.6426
std,0.2896,0.2259,0.2522
min,0.1146,0.0813,0.0000
25%,0.2929,0.6344,0.4987
50%,0.4598,0.7947,0.7488
75%,0.8744,0.8886,0.8189
max,0.9433,0.9375,0.9220


In [18]:
# Curvas de loss (treino vs validação)

hist_path = YOLOSEG_LOGS / "train_history.json"
if "train_history" not in vars():
    if not hist_path.exists():
        raise FileNotFoundError(
            f"Execute a célula de treino primeiro, ou verifique: {hist_path}")
    with open(hist_path) as f:
        train_history = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_history["train"], label="train")
ax.plot(train_history["val"],   label="val", ls="--")
ax.set_xlabel("Época")
ax.set_ylabel("Loss (CIoU + BCE_cls + DFL + BCE_mask)")
ax.set_title(f"YOLO-Seg ({YOLO_WEIGHTS}) BraTS2020 — {TRAIN_EPOCHS} épocas")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(YOLOSEG_LOGS / "loss_curves.png", dpi=150)
plt.show()

<Figure size 1000x400 with 1 Axes>

In [19]:
# Visualização adaptada para yolo GT vs PRED

cmap_mask = ListedColormap([(0,0,0,1), (1,0,0,1), (0,1,0,1), (0,0,1,1)])
norm_mask = BoundaryNorm([0, 1, 2, 3, 4], cmap_mask.N)

PLOT_MODS = YOLO_MODS


def masks_from_seg(seg):
    nec = seg == 1
    ede = seg == 2
    enh = seg == 3
    wt  = (seg == 1) | (seg == 2) | (seg == 3)
    tc  = (seg == 1) | (seg == 3)
    return nec, ede, enh, wt, tc


def overlay(ax, base2d, mask2d, color_rgb, alpha=0.45, title=""):
    ax.imshow(base2d, cmap="gray", origin="lower")

    rgba = np.zeros((*mask2d.shape, 4), dtype=np.float32)
    rgba[..., 0], rgba[..., 1], rgba[..., 2] = color_rgb
    rgba[..., 3] = mask2d.astype(np.float32) * alpha

    ax.imshow(rgba, origin="lower")
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def plot_random_case_multimodal_gt_pred(
        ids_list, split_name="test", seed=None, z=None,
        alpha_cls=0.45, alpha_comp=0.35):

    rng = np.random.default_rng(seed)
    cid = str(rng.choice(ids_list))
    d = case_dir(split_name, cid)

    gt = load_brats_seg(find_file(d, "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    if gt.shape != pr.shape:
        raise ValueError(
            f"Shape mismatch para {cid}: GT={gt.shape}, PRED={pr.shape}. "
            "Verifique a reconstrução 3D ou a orientação do NIfTI salvo."
        )

    if z is None:
        union = ((gt > 0) | (pr > 0)).astype(np.uint8)

        if union.any():
            z = pick_best_slice(union)
        else:
            z = pick_best_slice(gt)

    z = int(z)

    if z < 0 or z >= gt.shape[2]:
        raise ValueError(f"Slice z inválido: {z}. Volume possui D={gt.shape[2]}.")

    d1 = dice(gt == 1, pr == 1)
    d2 = dice(gt == 2, pr == 2)
    d3 = dice(gt == 3, pr == 3)
    wt = dice(((gt == 1) | (gt == 2) | (gt == 3)), ((pr == 1) | (pr == 2) | (pr == 3)))
    tc = dice(((gt == 1) | (gt == 3)), ((pr == 1) | (pr == 3)))

    print(f"\nPaciente: {cid} | split={split_name} | z={z}")
    print(f"Dice C1 (necrose/non-enh): {d1:.4f}")
    print(f"Dice C2 (edema):           {d2:.4f}")
    print(f"Dice ET (enhancing):       {d3:.4f}")
    print(f"Dice WT:                   {wt:.4f}")
    print(f"Dice TC:                   {tc:.4f}\n")

    gt2d = gt[:, :, z].T
    pr2d = pr[:, :, z].T

    gt_nec, gt_ede, gt_enh, gt_wt, gt_tc = masks_from_seg(gt2d)
    pr_nec, pr_ede, pr_enh, pr_wt, pr_tc = masks_from_seg(pr2d)

    fig, axes = plt.subplots(
        2 * len(PLOT_MODS),
        7,
        figsize=(28, 3.0 * 2 * len(PLOT_MODS))
    )

    if axes.ndim == 1:
        axes = axes.reshape(2 * len(PLOT_MODS), 7)

    fig.suptitle(f"YOLO-Seg | {cid} | z={z}", y=1.01)

    for i, mod in enumerate(PLOT_MODS):
        vol_mod = norm01(load_arr(find_file(d, mod)))
        img2d = vol_mod[:, :, z].T

        r_gt = 2 * i
        r_pr = 2 * i + 1

        axes[r_gt, 0].imshow(img2d, cmap="gray", origin="lower")
        axes[r_gt, 0].set_title(f"GT • {mod.upper()}", fontsize=8)
        axes[r_gt, 0].axis("off")

        axes[r_gt, 1].imshow(gt2d, cmap=cmap_mask, norm=norm_mask, origin="lower")
        axes[r_gt, 1].set_title("GT • Mask", fontsize=8)
        axes[r_gt, 1].axis("off")

        overlay(axes[r_gt, 2], img2d, gt_ede, (0,1,0), alpha_cls, "GT • Edema (2)")
        overlay(axes[r_gt, 3], img2d, gt_nec, (1,0,0), alpha_cls, "GT • Necrose (1)")
        overlay(axes[r_gt, 4], img2d, gt_enh, (0,0,1), alpha_cls, "GT • Enhancing (3)")
        overlay(axes[r_gt, 5], img2d, gt_wt,  (1,1,0), alpha_comp, "GT • Whole Tumor (WT)")
        overlay(axes[r_gt, 6], img2d, gt_tc,  (1,0,1), alpha_comp, "GT • Tumor Core (TC)")

        axes[r_pr, 0].imshow(img2d, cmap="gray", origin="lower")
        axes[r_pr, 0].set_title(f"PRED • {mod.upper()}", fontsize=8)
        axes[r_pr, 0].axis("off")

        axes[r_pr, 1].imshow(pr2d, cmap=cmap_mask, norm=norm_mask, origin="lower")
        axes[r_pr, 1].set_title("PRED • Mask", fontsize=8)
        axes[r_pr, 1].axis("off")

        overlay(axes[r_pr, 2], img2d, pr_ede, (0,1,0), alpha_cls, "PRED • Edema (2)")
        overlay(axes[r_pr, 3], img2d, pr_nec, (1,0,0), alpha_cls, "PRED • Necrose (1)")
        overlay(axes[r_pr, 4], img2d, pr_enh, (0,0,1), alpha_cls, "PRED • Enhancing (3)")
        overlay(axes[r_pr, 5], img2d, pr_wt,  (1,1,0), alpha_comp, "PRED • Whole Tumor (WT)")
        overlay(axes[r_pr, 6], img2d, pr_tc,  (1,0,1), alpha_comp, "PRED • Tumor Core (TC)")

    plt.tight_layout()

    out_fig = YOLOSEG_LOGS / f"qualitative_{cid}_z{z:03d}.png"
    fig.savefig(out_fig, dpi=150, bbox_inches="tight")

    print("Figura salva em:", out_fig)

    plt.show()
    plt.close(fig)

    return cid, z, {
        "dice_c1": d1,
        "dice_c2": d2,
        "dice_ET": d3,
        "dice_WT": wt,
        "dice_TC": tc
    }


cid, z, dice_dict = plot_random_case_multimodal_gt_pred(
    test_ids,
    split_name="test",
    seed=None
)


Paciente: BraTS20_Training_291 | split=test | z=88
Dice C1 (necrose/non-enh): 0.8133
Dice C2 (edema):           0.8411
Dice ET (enhancing):       0.0000
Dice WT:                   0.9368
Dice TC:                   0.8193

Figura salva em: D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\logs\qualitative_BraTS20_Training_291_z088.png


<Figure size 2800x1800 with 42 Axes>

In [20]:
# [1/6] Volume tumoral por caso

vol_rows = []
for cid in tqdm(test_ids, desc="Volume tumoral GT"):
    gt_path = find_file(case_dir("test", cid), "seg")
    gt = load_brats_seg(gt_path)
    spacing = nib.load(str(gt_path)).header.get_zooms()[:3]
    voxel_mm3 = float(np.prod(spacing))

    vol_rows.append({
        "id":         cid,
        "vol_WT_mm3": int(((gt==1)|(gt==2)|(gt==3)).sum()) * voxel_mm3,
        "vol_TC_mm3": int(((gt==1)|(gt==3)).sum())         * voxel_mm3,
        "vol_ET_mm3": int((gt==3).sum())                   * voxel_mm3,
    })

df_vol = pd.DataFrame(vol_rows)

df_err = (df.merge(df_hd95, on="id")
            .merge(lw_df,   on="id")
            .merge(df_vol,  on="id"))

df_err.describe().round(3)

Volume tumoral GT: 100%|██████████| 53/53 [00:02<00:00, 19.09it/s]


,dice_c1,dice_c2,dice_ET,dice_WT,dice_TC,hd95_c1,hd95_c2,hd95_ET,hd95_WT,hd95_TC,lwd_WT,lwd_TC,lwd_ET,vol_WT_mm3,vol_TC_mm3,vol_ET_mm3
count,53.000,53.000,53.000,53.000,53.000,45.000,53.000,52.000,53.000,53.000,53.000,53.000,53.000,53.000,53.000,53.000
mean,0.501,0.771,0.711,0.846,0.792,14.943,7.983,7.732,9.138,9.050,0.560,0.716,0.643,95115.849,38232.642,18346.698
std,0.335,0.124,0.204,0.093,0.157,20.360,12.659,18.127,12.435,16.652,0.290,0.226,0.252,57372.717,30648.866,15950.633
min,0.000,0.420,0.000,0.459,0.163,1.414,1.732,1.414,3.464,2.236,0.115,0.081,0.000,15762.000,1195.000,126.000
25%,0.149,0.671,0.660,0.821,0.765,5.000,3.000,2.000,5.385,4.243,0.293,0.634,0.499,51173.000,11695.000,4362.000
50%,0.598,0.816,0.787,0.867,0.844,8.775,4.583,2.236,6.782,5.916,0.460,0.795,0.749,80396.000,35839.000,17506.000
75%,0.777,0.865,0.839,0.902,0.896,16.363,6.708,3.281,8.062,8.062,0.874,0.889,0.819,135252.000,57282.000,29516.000
max,0.950,0.931,0.922,0.943,0.937,127.660,84.645,114.630,93.069,124.201,0.943,0.937,0.922,231951.000,153111.000,79009.000


In [21]:
# [2/6] Métricas agregadas por quartil de tamanho tumoral (quartiles do volume WT)

q25, q50, q75 = df_err["vol_WT_mm3"].quantile([0.25, 0.50, 0.75]).values

def size_bin(v):
    if v < q25: return "Q1"
    if v < q50: return "Q2"
    if v < q75: return "Q3"
    return "Q4"

df_err["size_bin"] = df_err["vol_WT_mm3"].apply(size_bin)

agg_cols = ["dice_WT", "dice_TC", "dice_ET",
            "hd95_WT", "hd95_TC", "hd95_ET",
            "lwd_WT",  "lwd_TC",  "lwd_ET"]

print(f"Limiares (vol WT em mm³): Q1 < {q25:,.0f}  |  Q2 < {q50:,.0f}  |  Q3 < {q75:,.0f}  |  Q4 ≥ {q75:,.0f}")
print(f"Contagem por quartil: {df_err['size_bin'].value_counts().to_dict()}\n")

df_err.groupby("size_bin")[agg_cols].mean().reindex(["Q1", "Q2", "Q3", "Q4"]).round(4)

Limiares (vol WT em mm³): Q1 < 51,173  |  Q2 < 80,396  |  Q3 < 135,252  |  Q4 ≥ 135,252
Contagem por quartil: {'Q4': 14, 'Q1': 13, 'Q2': 13, 'Q3': 13}



,dice_WT,dice_TC,dice_ET,hd95_WT,hd95_TC,hd95_ET,lwd_WT,lwd_TC,lwd_ET
size_bin,,,,,,,,,
Q1,0.7497,0.6953,0.7395,15.5557,14.0603,11.4838,0.4226,0.6528,0.6889
Q2,0.8648,0.8089,0.6724,6.7966,6.1074,7.2776,0.4911,0.7383,0.6560
Q3,0.8719,0.8688,0.7566,7.1903,6.6499,2.9179,0.6777,0.7974,0.6718
Q4,0.8926,0.7953,0.6772,7.1609,9.3573,9.1093,0.6425,0.6793,0.5601


In [22]:
# [3/6] Scatter: volume tumoral × Dice e × HD95 (por região)

regions = ["WT", "TC", "ET"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, reg in enumerate(regions):
    vol = df_err[f"vol_{reg}_mm3"].values
    dsc = df_err[f"dice_{reg}"].values
    hsc = df_err[f"hd95_{reg}"].values

    axes[0, j].scatter(vol, dsc, alpha=0.6, s=30)
    axes[0, j].set_xscale("log")
    axes[0, j].set_xlabel(f"Volume {reg} (mm³, log)")
    axes[0, j].set_ylabel("Dice")
    axes[0, j].set_title(f"{reg}: Volume × Dice")
    axes[0, j].grid(True, alpha=0.3)

    axes[1, j].scatter(vol, hsc, alpha=0.6, s=30, color="C1")
    axes[1, j].set_xscale("log")
    axes[1, j].set_xlabel(f"Volume {reg} (mm³, log)")
    axes[1, j].set_ylabel("HD95 (mm)")
    axes[1, j].set_title(f"{reg}: Volume × HD95")
    axes[1, j].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

<Figure size 1500x800 with 6 Axes>

In [23]:
# [3/6] Correlação de Spearman entre volume tumoral e métricas

corr_rows = []
for reg in regions:
    v = df_err[f"vol_{reg}_mm3"].values
    d = df_err[f"dice_{reg}"].values
    h = df_err[f"hd95_{reg}"].values

    rho_d, p_d = spearmanr(v, d)
    mask = ~np.isnan(h)
    if mask.sum() > 2:
        rho_h, p_h = spearmanr(v[mask], h[mask])
    else:
        rho_h, p_h = np.nan, np.nan

    corr_rows.append({
        "regiao": reg,
        "spearman_vol×dice": round(rho_d, 3),  "p_value_dice": round(p_d, 4),
        "spearman_vol×hd95": round(rho_h, 3),  "p_value_hd95": round(p_h, 4),
    })

print("Correlação de Spearman entre volume tumoral e métricas:")
pd.DataFrame(corr_rows)

Correlação de Spearman entre volume tumoral e métricas:


,regiao,spearman_vol×dice,p_value_dice,spearman_vol×hd95,p_value_hd95
0,WT,0.561,0.0000,-0.099,0.4798
1,TC,0.395,0.0034,0.354,0.0093
2,ET,0.549,0.0000,-0.278,0.0458


In [24]:
# [4/6] Ranking dos piores casos por Dice médio (WT/TC/ET)

K_WORST = 5

df_err["dice_mean"] = df_err[["dice_WT", "dice_TC", "dice_ET"]].mean(axis=1)
df_err["hd95_mean"] = df_err[["hd95_WT", "hd95_TC", "hd95_ET"]].mean(axis=1, skipna=True)

cols_show = ["id", "dice_WT", "dice_TC", "dice_ET",
             "hd95_WT", "hd95_TC", "hd95_ET",
             "vol_WT_mm3", "vol_ET_mm3", "size_bin"]

print(f"Top {K_WORST} piores casos por Dice médio:")
worst_dice = df_err.nsmallest(K_WORST, "dice_mean")[cols_show + ["dice_mean"]].round(3)
worst_ids = worst_dice["id"].tolist()

worst_dice

Top 5 piores casos por Dice médio:


,id,dice_WT,dice_TC,dice_ET,hd95_WT,hd95_TC,hd95_ET,vol_WT_mm3,vol_ET_mm3,size_bin,dice_mean
48,BraTS20_Training_316,0.776,0.163,0.403,9.110,21.401,14.560,231951.0,1585.0,Q4,0.447
4,BraTS20_Training_048,0.459,0.419,0.530,14.422,2.828,2.449,29239.0,2041.0,Q1,0.470
18,BraTS20_Training_271,0.936,0.769,0.000,3.606,8.062,NaN,51173.0,224.0,Q2,0.568
49,BraTS20_Training_291,0.937,0.819,0.000,6.782,6.403,16.621,218263.0,126.0,Q4,0.585
45,BraTS20_Training_301,0.874,0.599,0.297,6.000,20.905,29.210,173216.0,3873.0,Q4,0.590


In [25]:
# [4/6] Ranking dos piores casos por HD95 médio

print(f"Top {K_WORST} piores casos por HD95 médio:")
df_err.nlargest(K_WORST, "hd95_mean")[cols_show + ["hd95_mean"]].round(3)

Top 5 piores casos por HD95 médio:


,id,dice_WT,dice_TC,dice_ET,hd95_WT,hd95_TC,hd95_ET,vol_WT_mm3,vol_ET_mm3,size_bin,hd95_mean
1,BraTS20_Training_277,0.659,0.439,0.724,93.069,124.201,114.630,30073.0,1463.0,Q1,110.633
32,BraTS20_Training_292,0.892,0.896,0.331,4.123,3.000,62.312,77020.0,3815.0,Q2,23.145
45,BraTS20_Training_301,0.874,0.599,0.297,6.000,20.905,29.210,173216.0,3873.0,Q4,18.705
43,BraTS20_Training_308,0.815,0.520,0.511,12.728,16.763,21.256,202427.0,7070.0,Q4,16.916
48,BraTS20_Training_316,0.776,0.163,0.403,9.110,21.401,14.560,231951.0,1585.0,Q4,15.024


In [26]:
# [5/6] Visualização qualitativa dos piores casos

N_PLOT = min(1, len(worst_ids))

for cid in worst_ids[:N_PLOT]:
    plot_random_case_multimodal_gt_pred([cid], split_name="test", seed=0)


Paciente: BraTS20_Training_316 | split=test | z=90
Dice C1 (necrose/non-enh): 0.1366
Dice C2 (edema):           0.6489
Dice ET (enhancing):       0.4032
Dice WT:                   0.7756
Dice TC:                   0.1626

Figura salva em: D:\master_experiments\models_configs\YOLOSeg2D_BraTS2020\logs\qualitative_BraTS20_Training_316_z090.png


<Figure size 2800x1800 with 42 Axes>

In [27]:
# [6/6] Análise de componentes conexos: lesões perdidas (FN) vs (FP) por caso

CC_MIN_SIZE = 50

def count_components(gt_mask, pr_mask, min_size=CC_MIN_SIZE):
    structure = np.ones((3, 3, 3), dtype=int)
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)

    n_gt_valid = sum(1 for g in range(1, n_gt + 1) if (gt_lab == g).sum() >= min_size)
    n_pr_valid = sum(1 for p in range(1, n_pr + 1) if (pr_lab == p).sum() >= min_size)

    matched_pred, n_missed = set(), 0
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap = np.unique(pr_lab[gt_l]); overlap = overlap[overlap > 0]
        if len(overlap) == 0:
            n_missed += 1
        else:
            matched_pred.update(int(p) for p in overlap)

    n_spurious = sum(1 for p in range(1, n_pr + 1)
                     if p not in matched_pred and (pr_lab == p).sum() >= min_size)
    return n_gt_valid, n_pr_valid, n_missed, n_spurious


cc_rows = []
for cid in tqdm(test_ids, desc="Componentes conexos"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    nw = count_components((gt==1)|(gt==2)|(gt==3), (pr==1)|(pr==2)|(pr==3))
    nt = count_components((gt==1)|(gt==3),         (pr==1)|(pr==3))
    ne = count_components((gt==3),                  (pr==3))

    cc_rows.append({"id": cid,
        "WT_n_gt": nw[0], "WT_n_pr": nw[1], "WT_missed_FN": nw[2], "WT_spurious_FP": nw[3],
        "TC_n_gt": nt[0], "TC_n_pr": nt[1], "TC_missed_FN": nt[2], "TC_spurious_FP": nt[3],
        "ET_n_gt": ne[0], "ET_n_pr": ne[1], "ET_missed_FN": ne[2], "ET_spurious_FP": ne[3],
    })

df_cc = pd.DataFrame(cc_rows)

totals = pd.DataFrame({
    "região":          ["WT", "TC", "ET"],
    "tot_lesões_GT":   [df_cc["WT_n_gt"].sum(),       df_cc["TC_n_gt"].sum(),       df_cc["ET_n_gt"].sum()],
    "tot_lesões_PR":   [df_cc["WT_n_pr"].sum(),       df_cc["TC_n_pr"].sum(),       df_cc["ET_n_pr"].sum()],
    "tot_perdidas_FN": [df_cc["WT_missed_FN"].sum(),  df_cc["TC_missed_FN"].sum(),  df_cc["ET_missed_FN"].sum()],
    "tot_espúrias_FP": [df_cc["WT_spurious_FP"].sum(),df_cc["TC_spurious_FP"].sum(),df_cc["ET_spurious_FP"].sum()],
})
totals["FN_rate_%"] = (100 * totals["tot_perdidas_FN"] / totals["tot_lesões_GT"].replace(0, np.nan)).round(2)
totals["FP_rate_%"] = (100 * totals["tot_espúrias_FP"] / totals["tot_lesões_PR"].replace(0, np.nan)).round(2)

totals

Componentes conexos: 100%|██████████| 53/53 [00:49<00:00,  1.07it/s]


,região,tot_lesões_GT,tot_lesões_PR,tot_perdidas_FN,tot_espúrias_FP,FN_rate_%,FP_rate_%
0,WT,89,138,22,45,24.72,32.61
1,TC,63,74,5,8,7.94,10.81
2,ET,78,77,15,6,19.23,7.79


In [28]:
# [6/6] Top 10 casos com mais erros de componente

print("Top 10 casos com mais erros de componente (FN+FP somados em todas as regiões):")

df_cc["err_total"] = df_cc[["WT_missed_FN","TC_missed_FN","ET_missed_FN",
                            "WT_spurious_FP","TC_spurious_FP","ET_spurious_FP"]].sum(axis=1)

df_cc.nlargest(10, "err_total")

Top 10 casos com mais erros de componente (FN+FP somados em todas as regiões):


,id,WT_n_gt,WT_n_pr,WT_missed_FN,WT_spurious_FP,TC_n_gt,TC_n_pr,TC_missed_FN,TC_spurious_FP,ET_n_gt,ET_n_pr,ET_missed_FN,ET_spurious_FP,err_total
50,BraTS20_Training_175,4,3,2,1,4,1,3,0,4,1,3,0,9
24,BraTS20_Training_064,4,10,1,7,1,1,0,0,1,4,0,0,8
1,BraTS20_Training_277,3,4,2,2,1,3,0,1,2,3,0,1,6
48,BraTS20_Training_316,1,8,0,2,1,3,0,1,5,5,2,1,6
0,BraTS20_Training_110,5,6,2,3,1,2,0,0,1,2,0,0,5
32,BraTS20_Training_292,1,2,0,1,1,2,0,1,5,3,2,1,5
8,BraTS20_Training_119,4,6,2,2,1,1,0,0,1,1,0,0,4
35,BraTS20_Training_184,2,2,1,1,2,1,1,0,2,1,1,0,4
40,BraTS20_Training_151,2,4,0,2,1,3,0,2,1,1,0,0,4
43,BraTS20_Training_308,1,2,0,1,1,3,0,0,5,2,3,0,4
